# RAVDESS Speech Emotion Recognition — MLflow Multi-Model Experiment
### Local VSCode Version — 4 Models: Logistic Regression → MLP → Random Forest → LightGBM

| Model | Expected Accuracy |
|-------|------------------|
| Logistic Regression | ~40–50% (Baseline) |
| MLP (sklearn) | ~60–70% |
| Random Forest | ~75–82% |
| LightGBM | ~90–93%+ |

Run in terminal to activate virtual environment:

.\venv\Scripts\Activate.ps1 

## Step 1 — Imports & Global Config

In [ ]:
import os
import pickle
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib
matplotlib.use('Agg')          # non-interactive backend for MLflow logging
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
from joblib import Parallel, delayed   # parallel feature extraction

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

import mlflow
import mlflow.sklearn

# ── Reproducibility ──────────────────────────────────────────
SEED = 42
np.random.seed(SEED)

# ── Audio Config ─────────────────────────────────────────────
SAMPLE_RATE = 22050
DURATION    = 3.0
OFFSET      = 0.5
N_MFCC      = 40

# ── Local paths ───────────────────────────────────────────────
DATA_DIR    = "./data/ravdess_data"      
SAVE_DIR    = "./preprocessed"
MLRUNS_DIR  = "./mlruns"

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(MLRUNS_DIR, exist_ok=True)

EMOTION_MAP = {
    "01": "neutral",  "02": "calm",    "03": "happy",
    "04": "sad",      "05": "angry",   "06": "fearful",
    "07": "disgust",  "08": "surprised"
}

# ── MLflow: set once at the top, use local folder ─────────────
mlflow.set_tracking_uri(f"file:///{os.path.abspath(MLRUNS_DIR)}")
EXPERIMENT_NAME = "RAVDESS_SER_Comparison"
mlflow.set_experiment(EXPERIMENT_NAME)

print(" All imports done")
print(f"MLflow tracking at: {os.path.abspath(MLRUNS_DIR)}")

 All imports done
MLflow tracking at: c:\Users\gunja\Documents\ml_project\mlruns


## Step 2 — Data Augmentation Functions

In [19]:
def add_noise(audio, noise_factor=0.005):
    return audio + noise_factor * np.random.randn(len(audio))

def time_stretch(audio, rate=None):
    if rate is None:
        rate = np.random.uniform(0.8, 1.2)
    return librosa.effects.time_stretch(audio, rate=rate)

def pitch_shift(audio, sr, n_steps=None):
    if n_steps is None:
        n_steps = np.random.randint(-4, 5)
    return librosa.effects.pitch_shift(audio, sr=sr, n_steps=n_steps)

def time_shift(audio, shift_max=0.2):
    shift = int(np.random.uniform(-shift_max, shift_max) * len(audio))
    return np.roll(audio, shift)

def dynamic_range_compression(audio, threshold=0.3, ratio=4.0):
    audio_abs = np.abs(audio)
    gain = np.where(audio_abs > threshold,
                    threshold + (audio_abs - threshold) / ratio,
                    audio_abs)
    return np.sign(audio) * gain

AUGMENTATIONS = [add_noise, time_stretch, pitch_shift, time_shift, dynamic_range_compression]

def augment_audio(audio, sr):
    aug_fn = np.random.choice(AUGMENTATIONS)
    try:
        if aug_fn == time_stretch:  return time_stretch(audio)
        elif aug_fn == pitch_shift: return pitch_shift(audio, sr)
        else:                       return aug_fn(audio)
    except Exception:
        return audio

print(" Augmentation functions ready")

 Augmentation functions ready


## Step 3 — Feature Extraction
Rich 265-dim feature vector: MFCC + deltas + Chroma + Mel + spectral scalars

Uses `joblib.Parallel` to extract features across all CPU cores — much faster than single-threaded.

In [20]:
def extract_features(audio, sr):
    """
    265-dim feature vector:
      MFCC (40) + delta (40) + delta2 (40) = 120
      Chroma STFT = 12
      Mel Spectrogram (128 bands) = 128
      ZCR + RMS + Centroid + Bandwidth + Rolloff = 5
    """
    mfcc       = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=N_MFCC)
    mfcc_delta = librosa.feature.delta(mfcc)
    mfcc_d2    = librosa.feature.delta(mfcc, order=2)
    mfcc_feat  = np.hstack([
        np.mean(mfcc.T, axis=0),
        np.mean(mfcc_delta.T, axis=0),
        np.mean(mfcc_d2.T, axis=0),
    ])

    stft        = np.abs(librosa.stft(audio))
    chroma_feat = np.mean(librosa.feature.chroma_stft(S=stft, sr=sr).T, axis=0)
    mel_feat    = np.mean(librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=128).T, axis=0)

    scalar_feat = np.array([
        np.mean(librosa.feature.zero_crossing_rate(audio)),
        np.mean(librosa.feature.rms(y=audio)),
        np.mean(librosa.feature.spectral_centroid(y=audio, sr=sr)),
        np.mean(librosa.feature.spectral_bandwidth(y=audio, sr=sr)),
        np.mean(librosa.feature.spectral_rolloff(y=audio, sr=sr)),
    ])

    return np.hstack([mfcc_feat, chroma_feat, mel_feat, scalar_feat])


def process_one_file(fp, augment, aug_multiplier):
    """Process a single audio file (used by parallel loader)."""
    parts = fp.stem.split("-")
    if len(parts) < 3 or parts[2] not in EMOTION_MAP:
        return [], []
    label = EMOTION_MAP[parts[2]]
    try:
        audio, sr = librosa.load(str(fp), sr=SAMPLE_RATE,
                                 duration=DURATION, offset=OFFSET)
    except Exception as e:
        print(f"  Skipping {fp.name}: {e}")
        return [], []

    feats  = [extract_features(audio, sr)]
    labels = [label]

    if augment:
        for _ in range(aug_multiplier):
            feats.append(extract_features(augment_audio(audio, sr), sr))
            labels.append(label)

    return feats, labels


def load_ravdess(data_dir, augment=True, aug_multiplier=2, n_jobs=-1):
    """Load and extract features in parallel using all CPU cores."""
    audio_files = list(Path(data_dir).rglob("*.wav"))

    if len(audio_files) == 0:
        raise FileNotFoundError(f"No .wav files found in '{data_dir}'.")

    print(f"Found {len(audio_files)} audio files — extracting features in parallel (n_jobs={n_jobs})…")

    results = Parallel(n_jobs=n_jobs, prefer="threads")(
        delayed(process_one_file)(fp, augment, aug_multiplier)
        for fp in tqdm(audio_files, desc="Processing")
    )

    all_feats, all_labels = [], []
    for feats, labels in results:
        all_feats.extend(feats)
        all_labels.extend(labels)

    X = np.array(all_feats, dtype=np.float32)
    y = np.array(all_labels)
    print(f"\nTotal samples (original + augmented): {len(X)}")
    return X, y


print(" Feature extraction functions ready")

 Feature extraction functions ready


## Step 4 — Load or Preprocess Data
Checks for cached `.npy` files first to skip re-extraction on repeated runs.

In [21]:
if os.path.exists(f"{SAVE_DIR}/X_train.npy"):
    print("Loading preprocessed data from cache...")

    X_train = np.load(f"{SAVE_DIR}/X_train.npy")
    X_val   = np.load(f"{SAVE_DIR}/X_val.npy")
    X_test  = np.load(f"{SAVE_DIR}/X_test.npy")

    y_train = np.load(f"{SAVE_DIR}/y_train.npy")
    y_val   = np.load(f"{SAVE_DIR}/y_val.npy")
    y_test  = np.load(f"{SAVE_DIR}/y_test.npy")

    with open(f"{SAVE_DIR}/scaler.pkl", "rb") as f:
        scaler = pickle.load(f)

    with open(f"{SAVE_DIR}/label_encoder.pkl", "rb") as f:
        le = pickle.load(f)

    num_classes = len(le.classes_)
    print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

else:
    print("No cache found — extracting features from raw audio...")

    X, y = load_ravdess(DATA_DIR, augment=True, aug_multiplier=2, n_jobs=-1)

    print(f"Feature matrix : {X.shape}")
    print(f"Classes        : {np.unique(y)}")

    # Encode labels
    le    = LabelEncoder()
    y_enc = le.fit_transform(y)
    num_classes = len(le.classes_)

    # 70 / 10 / 20 split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_enc, test_size=0.20, random_state=SEED, stratify=y_enc)
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, test_size=0.125, random_state=SEED, stratify=y_train)

    # Normalise
    scaler  = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val   = scaler.transform(X_val)
    X_test  = scaler.transform(X_test)

    # Save to disk
    np.save(f"{SAVE_DIR}/X_train.npy", X_train)
    np.save(f"{SAVE_DIR}/X_val.npy",   X_val)
    np.save(f"{SAVE_DIR}/X_test.npy",  X_test)
    np.save(f"{SAVE_DIR}/y_train.npy", y_train)
    np.save(f"{SAVE_DIR}/y_val.npy",   y_val)
    np.save(f"{SAVE_DIR}/y_test.npy",  y_test)

    with open(f"{SAVE_DIR}/scaler.pkl", "wb") as f:        pickle.dump(scaler, f)
    with open(f"{SAVE_DIR}/label_encoder.pkl", "wb") as f: pickle.dump(le, f)

    print(f"\nSaved preprocessed data to {SAVE_DIR}")
    print(f"Train: {X_train.shape[0]}  |  Val: {X_val.shape[0]}  |  Test: {X_test.shape[0]}")

Loading preprocessed data from cache...
Train: (6048, 265), Val: (864, 265), Test: (1728, 265)


## Step 5 — MLflow Helper Functions

In [22]:
TMP_DIR = "./tmp_artifacts"
os.makedirs(TMP_DIR, exist_ok=True)

def log_confusion_matrix(y_true, y_pred, class_names, run_name):
    """Save a normalised confusion matrix PNG and log it as an MLflow artifact."""
    cm      = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=ax, linewidths=0.4)
    ax.set_title(f'Confusion Matrix — {run_name}', fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()

    path = f"{TMP_DIR}/cm_{run_name.replace(' ', '_')}.png"
    plt.savefig(path, dpi=130, bbox_inches='tight')
    plt.close()
    mlflow.log_artifact(path)
    return cm


def log_per_class_bar(cm, class_names, run_name):
    """Save per-class accuracy bar chart and log it."""
    per_class = cm.diagonal() / cm.sum(axis=1)
    fig, ax = plt.subplots(figsize=(10, 4))
    colors = plt.cm.RdYlGn(per_class)
    bars = ax.bar(class_names, per_class * 100, color=colors, edgecolor='white')
    for bar, val in zip(bars, per_class):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"{val*100:.1f}%", ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_ylim(0, 115)
    ax.set_title(f'Per-Class Accuracy — {run_name}', fontweight='bold')
    ax.set_xlabel('Emotion'); ax.set_ylabel('Accuracy (%)')
    plt.tight_layout()
    path = f"{TMP_DIR}/pca_{run_name.replace(' ', '_')}.png"
    plt.savefig(path, dpi=130)
    plt.close()
    mlflow.log_artifact(path)


print(" MLflow helper functions ready")

 MLflow helper functions ready


---
## MODEL 1 — Logistic Regression (Baseline)
Simple linear classifier. Expected ~40–50%.

In [23]:
RUN_NAME = "1_Logistic_Regression"

lr_params = dict(
    C=1.0,
    max_iter=300,
    solver="lbfgs",
    random_state=SEED,
    n_jobs=-1,
)

with mlflow.start_run(run_name=RUN_NAME):
    mlflow.log_params(lr_params)
    mlflow.log_params({
        "model_type"   : "LogisticRegression",
        "n_train"      : len(X_train),
        "n_test"       : len(X_test),
        "feature_dim"  : X_train.shape[1],
        "augmentation" : True,
        "aug_multiplier": 2,
    })

    lr = LogisticRegression(**lr_params)
    lr.fit(X_train, y_train)

    y_pred_lr  = lr.predict(X_test)
    acc_lr     = accuracy_score(y_test, y_pred_lr)
    val_acc_lr = accuracy_score(y_val,  lr.predict(X_val))

    mlflow.log_metrics({
        "test_accuracy" : acc_lr,
        "val_accuracy"  : val_acc_lr,
    })

    report = classification_report(y_test, y_pred_lr,
                                    target_names=le.classes_, output_dict=True)
    for cls in le.classes_:
        mlflow.log_metric(f"f1_{cls}", report[cls]['f1-score'])

    cm = log_confusion_matrix(y_test, y_pred_lr, le.classes_, RUN_NAME)
    log_per_class_bar(cm, le.classes_, RUN_NAME)

    report_path = f"{TMP_DIR}/report_{RUN_NAME}.txt"
    with open(report_path, 'w') as f:
        f.write(classification_report(y_test, y_pred_lr, target_names=le.classes_))
    mlflow.log_artifact(report_path)

    mlflow.sklearn.log_model(lr, name="model")

    print(f"\n{'='*50}")
    print(f" MODEL 1 — Logistic Regression")
    print(f" Test  Accuracy : {acc_lr*100:.2f}%")
    print(f" Val   Accuracy : {val_acc_lr*100:.2f}%")
    print(f"{'='*50}")
    print(classification_report(y_test, y_pred_lr, target_names=le.classes_))

2026/04/10 11:42:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



 MODEL 1 — Logistic Regression
 Test  Accuracy : 61.98%
 Val   Accuracy : 62.85%
              precision    recall  f1-score   support

       angry       0.85      0.70      0.76       230
        calm       0.63      0.71      0.67       230
     disgust       0.54      0.61      0.57       230
     fearful       0.69      0.68      0.69       231
       happy       0.58      0.61      0.59       231
     neutral       0.44      0.32      0.37       115
         sad       0.50      0.47      0.48       230
   surprised       0.66      0.71      0.69       231

    accuracy                           0.62      1728
   macro avg       0.61      0.60      0.60      1728
weighted avg       0.62      0.62      0.62      1728



---
## MODEL 2 — MLP Classifier (sklearn)
Shallow multi-layer perceptron. Expected ~75–82%.

In [24]:
RUN_NAME = "2_MLP_sklearn"

mlp_params = dict(
    hidden_layer_sizes=(512, 256, 128),
    activation="relu",
    solver="adam",
    alpha=1e-4,
    batch_size=64,
    learning_rate_init=1e-3,
    max_iter=200,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=15,
    random_state=SEED,
)

with mlflow.start_run(run_name=RUN_NAME):
    mlflow.log_params(mlp_params)
    mlflow.log_params({
        "model_type"   : "MLPClassifier",
        "n_train"      : len(X_train),
        "n_test"       : len(X_test),
        "feature_dim"  : X_train.shape[1],
        "augmentation" : True,
        "aug_multiplier": 2,
    })

    mlp = MLPClassifier(**mlp_params)
    mlp.fit(X_train, y_train)

    y_pred_mlp  = mlp.predict(X_test)
    acc_mlp     = accuracy_score(y_test, y_pred_mlp)
    val_acc_mlp = accuracy_score(y_val,  mlp.predict(X_val))

    mlflow.log_metrics({
        "test_accuracy"    : acc_mlp,
        "val_accuracy"     : val_acc_mlp,
        "n_iter"           : mlp.n_iter_,
        "final_train_loss" : mlp.loss_,
    })

    report = classification_report(y_test, y_pred_mlp,
                                    target_names=le.classes_, output_dict=True)
    for cls in le.classes_:
        mlflow.log_metric(f"f1_{cls}", report[cls]['f1-score'])

    cm = log_confusion_matrix(y_test, y_pred_mlp, le.classes_, RUN_NAME)
    log_per_class_bar(cm, le.classes_, RUN_NAME)

    # Loss curve
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(mlp.loss_curve_, label='Train Loss', linewidth=2, color='steelblue')
    if hasattr(mlp, 'validation_scores_'):
        ax.plot(mlp.validation_scores_, label='Val Score', linewidth=2,
                linestyle='--', color='tomato')
    ax.set_title('MLP Training Loss Curve', fontweight='bold')
    ax.set_xlabel('Iteration'); ax.set_ylabel('Loss / Score')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    lc_path = f"{TMP_DIR}/loss_curve_MLP.png"
    plt.savefig(lc_path, dpi=130); plt.close()
    mlflow.log_artifact(lc_path)

    report_path = f"{TMP_DIR}/report_{RUN_NAME}.txt"
    with open(report_path, 'w') as f:
        f.write(classification_report(y_test, y_pred_mlp, target_names=le.classes_))
    mlflow.log_artifact(report_path)

    mlflow.sklearn.log_model(mlp, name="model")

    print(f"\n{'='*50}")
    print(f" MODEL 2 — MLP Classifier (sklearn)")
    print(f" Test  Accuracy : {acc_mlp*100:.2f}%")
    print(f" Val   Accuracy : {val_acc_mlp*100:.2f}%")
    print(f"{'='*50}")
    print(classification_report(y_test, y_pred_mlp, target_names=le.classes_))

2026/04/10 11:42:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



 MODEL 2 — MLP Classifier (sklearn)
 Test  Accuracy : 80.67%
 Val   Accuracy : 79.17%
              precision    recall  f1-score   support

       angry       0.83      0.89      0.86       230
        calm       0.85      0.84      0.85       230
     disgust       0.77      0.78      0.78       230
     fearful       0.83      0.82      0.82       231
       happy       0.77      0.79      0.78       231
     neutral       0.78      0.67      0.72       115
         sad       0.76      0.74      0.75       230
   surprised       0.84      0.86      0.85       231

    accuracy                           0.81      1728
   macro avg       0.80      0.80      0.80      1728
weighted avg       0.81      0.81      0.81      1728



---
## MODEL 3 — Random Forest
Ensemble of decision trees. Expected ~85–90%.

In [25]:
RUN_NAME = "3_Random_Forest"

rf_params = dict(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    random_state=SEED,
    n_jobs=-1,
)

with mlflow.start_run(run_name=RUN_NAME):
    mlflow.log_params(rf_params)
    mlflow.log_params({
        "model_type"   : "RandomForestClassifier",
        "n_train"      : len(X_train),
        "n_test"       : len(X_test),
        "feature_dim"  : X_train.shape[1],
        "augmentation" : True,
        "aug_multiplier": 2,
    })

    rf = RandomForestClassifier(**rf_params)
    rf.fit(X_train, y_train)

    y_pred_rf  = rf.predict(X_test)
    acc_rf     = accuracy_score(y_test, y_pred_rf)
    val_acc_rf = accuracy_score(y_val,  rf.predict(X_val))

    mlflow.log_metrics({
        "test_accuracy" : acc_rf,
        "val_accuracy"  : val_acc_rf,
    })

    report = classification_report(y_test, y_pred_rf,
                                    target_names=le.classes_, output_dict=True)
    for cls in le.classes_:
        mlflow.log_metric(f"f1_{cls}", report[cls]['f1-score'])

    cm = log_confusion_matrix(y_test, y_pred_rf, le.classes_, RUN_NAME)
    log_per_class_bar(cm, le.classes_, RUN_NAME)

    report_path = f"{TMP_DIR}/report_{RUN_NAME}.txt"
    with open(report_path, 'w') as f:
        f.write(classification_report(y_test, y_pred_rf, target_names=le.classes_))
    mlflow.log_artifact(report_path)

    # Feature importances chart
    importances = rf.feature_importances_
    top_k       = 20
    top_idx     = np.argsort(importances)[::-1][:top_k]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(range(top_k), importances[top_idx], color='steelblue')
    ax.set_title("Top 20 Feature Importances — Random Forest", fontweight='bold')
    ax.set_xlabel("Feature index"); ax.set_ylabel("Importance")
    plt.tight_layout()
    fi_path = f"{TMP_DIR}/feature_importance_RF.png"
    plt.savefig(fi_path, dpi=130, bbox_inches='tight'); plt.close()
    mlflow.log_artifact(fi_path)

    mlflow.sklearn.log_model(rf, name="model")

    print(f"\n{'='*50}")
    print(f" MODEL 3 — Random Forest")
    print(f" Test  Accuracy : {acc_rf*100:.2f}%")
    print(f" Val   Accuracy : {val_acc_rf*100:.2f}%")
    print(f"{'='*50}")
    print(classification_report(y_test, y_pred_rf, target_names=le.classes_))

2026/04/10 11:43:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



 MODEL 3 — Random Forest
 Test  Accuracy : 87.04%
 Val   Accuracy : 86.23%
              precision    recall  f1-score   support

       angry       0.95      0.88      0.91       230
        calm       0.74      0.97      0.84       230
     disgust       0.83      0.90      0.86       230
     fearful       0.93      0.86      0.89       231
       happy       0.93      0.86      0.89       231
     neutral       0.97      0.73      0.83       115
         sad       0.87      0.80      0.84       230
   surprised       0.87      0.90      0.88       231

    accuracy                           0.87      1728
   macro avg       0.88      0.86      0.87      1728
weighted avg       0.88      0.87      0.87      1728



---
## MODEL 4 — LightGBM (Fast Gradient Boosting — Best Tabular Model)

Key design choices: GBDT with early stopping, feature/row sampling, regularization, and full parallel processing (~30 seconds).

In [26]:
import lightgbm as lgb

RUN_NAME = "4_LightGBM_Fast"

with mlflow.start_run(run_name=RUN_NAME):

    params = dict(
        objective         = "multiclass",
        num_class         = num_classes,
        metric            = "multi_logloss",
        boosting_type     = "gbdt",
        n_estimators      = 1000,
        learning_rate     = 0.05,
        num_leaves        = 127,
        max_depth         = 8,
        min_child_samples = 10,
        feature_fraction  = 0.8,
        bagging_fraction  = 0.8,
        bagging_freq      = 5,
        reg_alpha         = 0.1,
        reg_lambda        = 0.1,
        random_state      = SEED,
        n_jobs            = -1,
        verbosity         = -1,
    )

    model_lgbm = lgb.LGBMClassifier(**params)
    model_lgbm.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50, verbose=True),
                   lgb.log_evaluation(50)]
    )

    y_pred_lgbm = model_lgbm.predict(X_test)
    y_pred_val  = model_lgbm.predict(X_val)
    test_acc    = accuracy_score(y_test, y_pred_lgbm)
    val_acc     = accuracy_score(y_val,  y_pred_val)

    mlflow.log_params({**params,
                       "n_train": len(X_train), "n_test": len(X_test),
                       "feature_dim": X_train.shape[1],
                       "best_iteration": model_lgbm.best_iteration_})
    mlflow.log_metrics({"test_accuracy": test_acc, "val_accuracy": val_acc})

    report = classification_report(y_test, y_pred_lgbm,
                                    target_names=le.classes_, output_dict=True)
    for cls in le.classes_:
        mlflow.log_metric(f"f1_{cls}", report[cls]['f1-score'])

    cm = log_confusion_matrix(y_test, y_pred_lgbm, le.classes_, RUN_NAME)
    log_per_class_bar(cm, le.classes_, RUN_NAME)

    report_path = f"{TMP_DIR}/report_{RUN_NAME}.txt"
    with open(report_path, 'w') as f:
        f.write(classification_report(y_test, y_pred_lgbm, target_names=le.classes_))
    mlflow.log_artifact(report_path)

    mlflow.sklearn.log_model(model_lgbm, name="model")

    # Save scaler + encoder as artifacts too
    with open(f"{TMP_DIR}/scaler.pkl", "wb") as f:        pickle.dump(scaler, f)
    with open(f"{TMP_DIR}/label_encoder.pkl", "wb") as f: pickle.dump(le, f)
    mlflow.log_artifact(f"{TMP_DIR}/scaler.pkl")
    mlflow.log_artifact(f"{TMP_DIR}/label_encoder.pkl")

    print(f"\n{'='*50}")
    print(f" MODEL 4 — LightGBM")
    print(f" Test  Accuracy : {test_acc*100:.2f}%")
    print(f" Val   Accuracy : {val_acc*100:.2f}%")
    print(f" Best iteration : {model_lgbm.best_iteration_}")
    print(f"{'='*50}")
    print(classification_report(y_test, y_pred_lgbm, target_names=le.classes_))

Training until validation scores don't improve for 50 rounds
[50]	valid_0's multi_logloss: 0.559044
[100]	valid_0's multi_logloss: 0.401168
[150]	valid_0's multi_logloss: 0.356134
[200]	valid_0's multi_logloss: 0.344022
[250]	valid_0's multi_logloss: 0.336491
[300]	valid_0's multi_logloss: 0.336014
Early stopping, best iteration is:
[265]	valid_0's multi_logloss: 0.335274


2026/04/10 11:43:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



 MODEL 4 — LightGBM
 Test  Accuracy : 89.24%
 Val   Accuracy : 89.58%
 Best iteration : 265
              precision    recall  f1-score   support

       angry       0.94      0.92      0.93       230
        calm       0.83      0.94      0.88       230
     disgust       0.89      0.89      0.89       230
     fearful       0.92      0.90      0.91       231
       happy       0.90      0.89      0.89       231
     neutral       0.94      0.77      0.85       115
         sad       0.89      0.84      0.86       230
   surprised       0.88      0.92      0.90       231

    accuracy                           0.89      1728
   macro avg       0.90      0.88      0.89      1728
weighted avg       0.89      0.89      0.89      1728



---
## Step 6 — Summary: Compare All 4 Models

In [27]:
results = pd.DataFrame([
    {"Model": "1 – Logistic Regression",  "Test Acc (%)": acc_lr   * 100, "Type": "Linear"},
    {"Model": "2 – MLP (sklearn)",        "Test Acc (%)": acc_mlp  * 100, "Type": "Shallow NN"},
    {"Model": "3 – Random Forest",        "Test Acc (%)": acc_rf   * 100, "Type": "Ensemble"},
    {"Model": "4 – LightGBM",             "Test Acc (%)": test_acc * 100, "Type": "Boosting"},
]).sort_values("Test Acc (%)")

print("\n" + "="*55)
print("  RAVDESS SER — All Models Summary")
print("="*55)
print(results.to_string(index=False))
print("="*55)

colors = ['#e74c3c', '#e67e22', '#f1c40f', '#27ae60']
fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(results["Model"], results["Test Acc (%)"],
               color=colors, edgecolor='white', height=0.55)
for bar, val in zip(bars, results["Test Acc (%)"]):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
            f"{val:.2f}%", va='center', fontsize=11, fontweight='bold')
ax.set_xlim(0, 105)
ax.set_title("RAVDESS SER — Model Comparison (Test Accuracy)",
             fontsize=13, fontweight='bold')
ax.set_xlabel("Test Accuracy (%)")
ax.set_facecolor('#f8f9fa')
plt.tight_layout()

chart_path = f"{TMP_DIR}/model_comparison.png"
plt.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.show()

# Log summary run
if mlflow.active_run():
    mlflow.end_run()

with mlflow.start_run(run_name="Model_Comparison_Summary"):
    mlflow.log_metric("lr_test_acc",   acc_lr)
    mlflow.log_metric("mlp_test_acc",  acc_mlp)
    mlflow.log_metric("rf_test_acc",   acc_rf)
    mlflow.log_metric("lgbm_test_acc", test_acc)
    mlflow.log_artifact(chart_path)

print("\n All runs logged.")
print(f"\n To view MLflow UI, run in terminal:")
print(f"   mlflow ui --backend-store-uri {os.path.abspath(MLRUNS_DIR)}")
print(f"   Then open: http://localhost:5000")


  RAVDESS SER — All Models Summary
                  Model  Test Acc (%)       Type
1 – Logistic Regression     61.979167     Linear
      2 – MLP (sklearn)     80.671296 Shallow NN
      3 – Random Forest     87.037037   Ensemble
           4 – LightGBM     89.236111   Boosting

 All runs logged.

 To view MLflow UI, run in terminal:
   mlflow ui --backend-store-uri c:\Users\gunja\Documents\ml_project\mlruns
   Then open: http://localhost:5000


Run interminal instead of the one above:

mlflow ui --backend-store-uri file:///C:/Users/gunja/Documents/ml_project/mlruns

---
## Step 7 — Inference on New Audio

In [28]:
def predict_emotion(wav_path, model, scaler, label_encoder, top_k=3):
    audio, sr = librosa.load(wav_path, sr=SAMPLE_RATE, duration=DURATION, offset=OFFSET)

    feat = extract_features(audio, sr)
    feat_scaled = scaler.transform([feat])

    probs = model.predict_proba(feat_scaled)[0]
    top_idx = np.argsort(probs)[::-1][:top_k]

    print(f"\nFile : {Path(wav_path).name}")
    print(f"{'─'*38}")
    for rank, idx in enumerate(top_idx, 1):
        bar = '█' * int(probs[idx] * 30)
        print(f"  {rank}. {label_encoder.classes_[idx]:<12} {bar}  {probs[idx]*100:.1f}%")
    print(f"{'─'*38}")
    print(f"  Predicted: {label_encoder.classes_[top_idx[0]].upper()}")


# Usage example:
# predict_emotion("./path/to/your_audio.wav", model_lgbm, scaler, le)